In [4]:
import sys, os
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
DATA = ROOT / "Evaluator" / "Testing_data"
CAND_DIR = DATA / "folder_for_ranking"
GT_PATH = DATA / "cea_gt.csv"

sys.path.insert(0, str(ROOT / "Pipeline_semtab" / "Ranking"))
from data_loader import load_candidates, group_by_cell, tab_id_from_filename
from cta import cta_from_cea
from cea import build_type_pct, DEFAULT_WEIGHTS
from scoring import Scoring_method

DEFAULT_WEIGHTS

(0.5, 0.2, 0.3)

In [5]:
scorer = Scoring_method()

gt = pd.read_csv(GT_PATH, header=None, names=["tab", "row", "col", "uri"], dtype=str)
gold_by_cell = {(t, int(r), int(c)): {u.split("/")[-1].lower() for u in uris.split()}for t, r, c, uris in gt.itertuples(index=False)}

cells = []
for filename in sorted(f for f in os.listdir(CAND_DIR) if f.endswith(".csv")):
    cand_df = load_candidates(CAND_DIR / filename)
    tab = tab_id_from_filename(filename)
    type_pct = build_type_pct(cta_from_cea(cand_df))
    for (r, c), cands in group_by_cell(cand_df).items():
        gold = gold_by_cell.get((tab, r + 1, c))
        if gold is None:
            continue
        features = []
        for cand in cands:
            if not cand["qid"]:
                continue
            aliases = cand["aliases"].split("|") if cand["aliases"] else None
            features.append((scorer.best_string_sim(cand["mention"], cand["label"], aliases),scorer.quality_score(cand["quality"]),scorer.type_coherence(cand["P31"] + cand["P279"], type_pct.get(c, {})),str(cand["qid"]).lower() in gold))
        if features:
            cells.append(features)

print(f"Evaluable cells: {len(cells)}")
print(f"Candidates: {sum(len(f) for f in cells)}")
print(f"Single-candidate: {sum(1 for f in cells if len(f) == 1)}")
print(f"Gold among candidates: {sum(1 for f in cells if any(g for *_, g in f)) / len(cells):.1%}")

Evaluable cells: 4144
Candidates: 171509
Single-candidate: 409
Gold among candidates: 93.8%


In [6]:
def rank(cell, weights):
    a, b, c = weights
    scores = [a * sim + b * qual + c * tc for sim, qual, tc, _ in cell]
    order = sorted(range(len(cell)), key=lambda i: scores[i], reverse=True)
    return [(scores[i], cell[i][3]) for i in order]

## 2. Weights

In [7]:
SETTINGS = {
    "default (0.5, 0.2, 0.3)":(0.5, 0.2, 0.3),
    "similarity only (1, 0, 0)":(1.0, 0.0, 0.0),
    "quality only (0, 1, 0)":(0.0, 1.0, 0.0),
    "type only (0, 0, 1)":(0.0, 0.0, 1.0),
    "uniform (1/3, 1/3, 1/3)":(1 / 3, 1 / 3, 1 / 3),
    "no type (0.7, 0.3, 0)":(0.7, 0.3, 0.0),
    "no quality (0.6, 0, 0.4)":(0.6, 0.0, 0.4),
    "more type (0.4, 0.2, 0.4)":(0.4, 0.2, 0.4),
    "less type (0.6, 0.2, 0.2)":(0.6, 0.2, 0.2)
}

rows = []
for name, weights in SETTINGS.items():
    correct = 0
    below = 0
    for cell in cells:
        ranked = rank(cell, weights)
        correct += ranked[0][1]
        if len(ranked) > 1 and ranked[0][0] - ranked[1][0] < 0.10:
            below += 1
    rows.append({"weights": name, "top-1 accuracy": round(correct / len(cells), 4),"cells below margin 0.10": below})

results = pd.DataFrame(rows).sort_values("top-1 accuracy", ascending=False)
results

,weights,top-1 accuracy,cells below margin 0.10
8,"less type (0.6, 0.2, 0.2)",0.7288,2542
0,"default (0.5, 0.2, 0.3)",0.7276,2244
4,"uniform (1/3, 1/3, 1/3)",0.7230,2259
6,"no quality (0.6, 0, 0.4)",0.7218,2209
7,"more type (0.4, 0.2, 0.4)",0.7208,2194
3,"type only (0, 0, 1)",0.6561,2618
5,"no type (0.7, 0.3, 0)",0.6540,2847
1,"similarity only (1, 0, 0)",0.6530,2866
2,"quality only (0, 1, 0)",0.6433,3185


In [ ]:
default_acc = results.loc[results.weights.str.startswith("default"), "top-1 accuracy"].iloc[0]
best = results.iloc[0]
sigma = (default_acc * (1 - default_acc) / len(cells)) ** 0.5
retrieval_order = sum(cell[0][3] for cell in cells) / len(cells)

print(f"Default: {default_acc:.4f}")
print(f"Best tested: {best['top-1 accuracy']:.4f}  ({best['weights']})")
print(f"Difference: {100 * (best['top-1 accuracy'] - default_acc):+.2f} pt(1 sigma = {100 * sigma:.2f} pt)")
print(f"Worst tested: {results['top-1 accuracy'].min():.4f}  ({results.iloc[-1]['weights']})")
print(f"Retrieval order: {retrieval_order:.4f}   score gain: {100 * (default_acc - retrieval_order):+.2f} pt")

Default: 0.7276
Best tested: 0.7288  (less type (0.6, 0.2, 0.2))
Difference: +0.12 pt   (1 sigma = 0.69 pt)
Worst tested: 0.6433  (quality only (0, 1, 0))
Retrieval order: 0.6433   score gain: +8.43 pt


## 3. Margin

In [9]:
gaps = []
for cell in cells:
    ranked = rank(cell, DEFAULT_WEIGHTS)
    if len(ranked) > 1:
        gaps.append((ranked[0][0] - ranked[1][0], ranked[0][1]))

rows = []
for m in [0.01, 0.05, 0.10, 0.20, 0.30, 0.50]:
    below = [ok for gap, ok in gaps if gap < m]
    rows.append({"margin": m,"cells below": len(below), "% of cells": round(100 * len(below) / len(cells), 1),"top-1 already correct": sum(below),"top-1 wrong": len(below) - sum(below)})
pd.DataFrame(rows)

,margin,cells below,% of cells,top-1 already correct,top-1 wrong
0,0.01,1454,35.1,726,728
1,0.05,1821,43.9,929,892
2,0.10,2244,54.2,1260,984
3,0.20,2873,69.3,1802,1071
4,0.30,3246,78.3,2154,1092
5,0.50,3551,85.7,2450,1101


In [11]:
print(f"Evaluable cells:{len(cells)}")
print(f"More than one candidate: {len(gaps)}")
print(f"Exact ties (gap = 0): {sum(1 for g, _ in gaps if g == 0)}\n")

for m, name in [(0.05, "CEA_TIEBREAK_MARGIN (limited_slm)"), (0.10, "LLM_CONTEXT_MARGIN (slm_context)")]:
    below = [ok for gap, ok in gaps if gap < m]
    print(f"Margin {m:.2f}-{name}")
    print(f"{len(below)} cells below ({100 * len(below) / len(cells):.1f} % of all cells) = {len(below)} SLM calls")
    print(f"{sum(below)} with top-1 already correct, {len(below) - sum(below)} with top-1 wrong\n")

Evaluable cells:4144
More than one candidate: 3735
Exact ties (gap = 0): 1440

Margin 0.05-CEA_TIEBREAK_MARGIN (limited_slm)
1821 cells below (43.9 % of all cells) = 1821 SLM calls
929 with top-1 already correct, 892 with top-1 wrong

Margin 0.10-LLM_CONTEXT_MARGIN (slm_context)
2244 cells below (54.2 % of all cells) = 2244 SLM calls
1260 with top-1 already correct, 984 with top-1 wrong

